Aquí tienes el script adaptado. El diseño estructural incluye la segmentación dinámica para guardar el dataset reducido de Lasso y un bucle que segmenta los años de inicio de entrenamiento según lo solicitado ($2021, 2022, 2023, 2024 \text{ y } 2025$), manteniendo siempre como límite superior los registros disponibles del año 2026.




In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error
import warnings

# Silenciar advertencias de convergencia de forma limpia
warnings.filterwarnings("ignore")

# 1. Configuración de rutas de archivos y directorios
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\arima\06_entrenar_modelo\0_obtener_rezagos\2_datos\1_raw\meteo_epi_2021-2026_1.xlsx"
directorio_procesados = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\2_datos\2_procesados"
directorio_resultados = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados"

os.makedirs(directorio_procesados, exist_ok=True)
os.makedirs(directorio_resultados, exist_ok=True)

# 2. Cargar datos y ordenar cronológicamente
print("Cargando y preparando el dataset original...")
df = pd.read_excel(ruta_entrada)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# 3. Generar todos los 12 rezagos originales (para buscar el filtrado Lasso)
print("Generando la matriz completa de rezagos...")
columnas_excluidas = ['fecha', 'semana_epi', 'año']
columnas_con_rezago = [col for col in df.columns if col not in columnas_excluidas]

nuevas_columnas = {}
for col in columnas_con_rezago:
    for lag in range(1, 13):
        nuevas_columnas[f"{col}_lag_{lag}"] = df[col].shift(lag)

df_rezagos = pd.DataFrame(nuevas_columnas, index=df.index)

# CORRECCIÓN DE COLUMNAS REPETIDAS: Unir asegurando limpiar cualquier duplicado latente
df_procesado = pd.concat([df, df_rezagos], axis=1).dropna()
df_procesado = df_procesado.loc[:, ~df_procesado.columns.duplicated()].reset_index(drop=True)

# Separar variables predictoras crudas y objetivo para Lasso
columnas_candidatas = [col for col in df_procesado.columns if col not in ['fecha', 'casos_dengue', 'semana_epi', 'año']]
X_cruda = df_procesado[columnas_candidatas]
y_cruda = df_procesado['casos_dengue']

# 4. Selección de Variables mediante Regresión Lasso (con Validación Cruzada)
print("Ejecutando LassoCV para identificar rezagos y variables clave...")
scaler = StandardScaler()
X_cruda_escalada = scaler.fit_transform(X_cruda)

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_cruda_escalada, y_cruda)

coeficientes = lasso.coef_
columnas_seleccionadas = [col for col, coef in zip(columnas_candidatas, coeficientes) if coef != 0]
print(f"-> Lasso redujo tus predictores de {X_cruda.shape[1]} a {len(columnas_seleccionadas)} variables clave.")

# 5. Guardar el Dataset Reducido solicitado
columnas_finales_guardar = ['fecha', 'año', 'semana_epi', 'casos_dengue'] + columnas_seleccionadas
df_reducido = df_procesado[columnas_finales_guardar].copy().reset_index(drop=True)

ruta_dataset_reducido = os.path.join(directorio_procesados, "dataset_reducido_lasso.xlsx")
df_reducido.to_excel(ruta_dataset_reducido, index=False)
print(f"Dataset reducido guardado exitosamente en: {ruta_dataset_reducido}")

# =============================================================================
# 6. CICLO EVALUATIVO POR VENTANA TEMPORAL (Análisis de Sensibilidad)
# =============================================================================
años_inicio = [2021, 2022, 2023, 2024, 2025]
metricas_experimentos = []
experimento_base_grafico = {}

for anio in años_inicio:
    print(f"\n========================================================")
    print(f"Iniciando Ventana: Desde {anio} hasta registros de 2026")
    print(f"========================================================")
    
    # CORRECCIÓN DE LA CORRECCIÓN: Filtrado seguro forzando la evaluación a un vector 1D (.iloc[:, 0] si persistiera)
    # Al limpiar las columnas duplicadas en el paso 3, este filtro funciona de forma directa y nativa sin colapsar.
    df_ventana = df_reducido[df_reducido['año'] >= anio].reset_index(drop=True)
    
    X_filtrada = df_ventana[columnas_seleccionadas]
    y = df_ventana['casos_dengue']
    fechas = df_ventana['fecha']
    
    # Estrategia Rolling Forecast (Testeo en el último 20% de la ventana actual)
    tamanio_entrenamiento_inicial = int(len(df_ventana) * 0.80)
    predicciones_test = []
    mae_entrenamiento_lista = []
    indices_test = range(tamanio_entrenamiento_inicial, len(df_ventana))
    
    orden_arima = (1, 1, 0)
    
    for i in indices_test:
        X_train_fold = X_filtrada.iloc[:i]
        y_train_fold = y.iloc[:i]
        X_test_fold = X_filtrada.iloc[i:i+1]
        
        modelo = ARIMA(endog=y_train_fold, exog=X_train_fold, order=orden_arima)
        modelo_ajustado = modelo.fit(method_kwargs={'maxiter': 150})
        
        pred_train_fold = modelo_ajustado.fittedvalues
        mae_train_paso = mean_absolute_error(y_train_fold[1:], pred_train_fold[1:])
        mae_entrenamiento_lista.append(mae_train_paso)
        
        pred_test_paso = modelo_ajustado.forecast(steps=1, exog=X_test_fold)
        predicciones_test.append(pred_test_paso.values[0])
        
    # Consolidar métricas de la ventana actual
    y_test_real = y.iloc[tamanio_entrenamiento_inicial:].values
    predicciones_test = np.array(predicciones_test)
    fechas_test = fechas.iloc[tamanio_entrenamiento_inicial:].reset_index(drop=True)
    
    mae_test_global = mean_absolute_error(y_test_real, predicciones_test)
    mae_train_global = np.mean(mae_entrenamiento_lista)
    
    print(f"MAE Promedio Entrenamiento ({anio}-2026): {mae_train_global:.4f}")
    print(f"MAE Global Testeo ({anio}-2026): {mae_test_global:.4f}")
    
    metricas_experimentos.append({
        'Ventana_Inicio': f"{anio}-2026",
        'MAE_Entrenamiento': mae_train_global,
        'MAE_Testeo': mae_test_global
    })
    
    if anio == 2021:
        experimento_base_grafico = {
            'fechas_train': fechas.iloc[:tamanio_entrenamiento_inicial],
            'y_train': y.iloc[:tamanio_entrenamiento_inicial],
            'X_train': X_filtrada.iloc[:tamanio_entrenamiento_inicial],
            'fechas_test': fechas_test,
            'y_test_real': y_test_real,
            'predicciones_test': predicciones_test,
            'mae_train': mae_train_global,
            'mae_test': mae_test_global
        }

# =============================================================================
# 7. Exportación de Resultados Multiventana a Excel
# =============================================================================
df_resumen_ventanas = pd.DataFrame(metricas_experimentos)
ruta_excel = os.path.join(directorio_resultados, "desempeno_rolling_forecast_ventanas.xlsx")

with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    df_resumen_ventanas.to_excel(writer, sheet_name='Comparativa_Ventanas', index=False)
    df_series_base = pd.DataFrame({
        'Fecha': experimento_base_grafico['fechas_test'],
        'Casos_Reales': experimento_base_grafico['y_test_real'],
        'Casos_Predichos': experimento_base_grafico['predicciones_test']
    })
    df_series_base.to_excel(writer, sheet_name='Predicciones_Base_2021_2026', index=False)

print(f"\n[OK] Cuadro comparativo multiventana guardado en Excel: {ruta_excel}")

# =============================================================================
# 8. Construcción de Gráficas Requeridas (Líneas y Barras)
# =============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

modelo_final = ARIMA(
    endog=experimento_base_grafico['y_train'], 
    exog=experimento_base_grafico['X_train'], 
    order=orden_arima
).fit(method_kwargs={'maxiter': 150})

ax1.plot(experimento_base_grafico['fechas_train'], experimento_base_grafico['y_train'], label='Real (Entrenamiento 2021+)', color='darkblue', alpha=0.7)
ax1.plot(experimento_base_grafico['fechas_train'], modelo_final.fittedvalues, label='Ajustado ARIMAX (Lasso)', color='orange', linestyle='--', alpha=0.8)
ax1.set_title(f'Desempeño en Entrenamiento\n(MAE Promedio: {experimento_base_grafico["mae_train"]:.2f})')
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos de Dengue')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

ax2.plot(experimento_base_grafico['fechas_test'], experimento_base_grafico['y_test_real'], label='Real (Testeo Móvil)', color='darkgreen', marker='o', markersize=3)
ax2.plot(experimento_base_grafico['fechas_test'], experimento_base_grafico['predicciones_test'], label='Predicción Rolling Forecast', color='red', linestyle='--', marker='x', markersize=3)
ax2.set_title(f'Desempeño en Testeo Móvil (Histórico completo)\n(MAE Global: {experimento_base_grafico["mae_test"]:.2f})')
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos de Dengue')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(directorio_resultados, "comparativa_lineas_desempeno.png"), dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
x_indexes = np.arange(len(df_resumen_ventanas))
width = 0.35

barras_train = plt.bar(x_indexes - width/2, df_resumen_ventanas['MAE_Entrenamiento'], width, label='MAE Entrenamiento', color='#4C72B0', edgecolor='black')
barras_test = plt.bar(x_indexes + width/2, df_resumen_ventanas['MAE_Testeo'], width, label='MAE Testeo', color='#C44E52', edgecolor='black')

plt.xlabel('Ventana Temporal de Entrada Utilizada')
plt.ylabel('Error Absoluto Medio (MAE)')
plt.title('Impacto del Horizonte Inicial en la Capacidad Predictiva del Modelo')
plt.xticks(x_indexes, df_resumen_ventanas['Ventana_Inicio'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()

for b_tr, b_ts in zip(barras_train, barras_test):
    plt.text(b_tr.get_x() + b_tr.get_width()/2.0, b_tr.get_height() + 0.1, f"{b_tr.get_height():.2f}", ha='center', va='bottom', fontsize=9)
    plt.text(b_ts.get_x() + b_ts.get_width()/2.0, b_ts.get_height() + 0.1, f"{b_ts.get_height():.2f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(directorio_resultados, "comparativa_barras_mae.png"), dpi=300)
plt.close()

print("\n¡Ejecución Multiventana finalizada! El conflicto multidimensional quedó completamente resuelto.")

Cargando y preparando el dataset original...
Generando la matriz completa de rezagos...
Ejecutando LassoCV para identificar rezagos y variables clave...
-> Lasso redujo tus predictores de 168 a 14 variables clave.
Dataset reducido guardado exitosamente en: C:\Users\marco\Documentos\investigacion\rolling_forecast\2_datos\2_procesados\dataset_reducido_lasso.xlsx

Iniciando Ventana: Desde 2021 hasta registros de 2026
MAE Promedio Entrenamiento (2021-2026): 5.6829
MAE Global Testeo (2021-2026): 7.9513

Iniciando Ventana: Desde 2022 hasta registros de 2026
MAE Promedio Entrenamiento (2022-2026): 6.3506
MAE Global Testeo (2022-2026): 7.7252

Iniciando Ventana: Desde 2023 hasta registros de 2026
MAE Promedio Entrenamiento (2023-2026): 7.2584
MAE Global Testeo (2023-2026): 8.1769

Iniciando Ventana: Desde 2024 hasta registros de 2026
MAE Promedio Entrenamiento (2024-2026): 8.2060
MAE Global Testeo (2024-2026): 13.0292

Iniciando Ventana: Desde 2025 hasta registros de 2026
MAE Promedio Entrenam

# Visualización gráfica de los resultados de los entrenamientos anteriores

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings

# Silenciar advertencias de convergencia
warnings.filterwarnings("ignore")

# 1. Configuración de rutas (Cargar el dataset reducido guardado por Lasso anteriormente)
ruta_dataset_reducido = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\2_datos\2_procesados\dataset_reducido_lasso.xlsx"
directorio_resultados = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados"
os.makedirs(directorio_resultados, exist_ok=True)

if not os.path.exists(ruta_dataset_reducido):
    raise FileNotFoundError(f"No se encontró el dataset reducido en {ruta_dataset_reducido}. Corre primero el script principal de Lasso.")

# 2. Leer datos preparados
df_reducido = pd.read_excel(ruta_dataset_reducido)
df_reducido['fecha'] = pd.to_datetime(df_reducido['fecha'])

# Identificar cuáles son las variables exógenas (excluyendo metadatos y target)
columnas_exogenas = [col for col in df_reducido.columns if col not in ['fecha', 'año', 'semana_epi', 'casos_dengue']]

# 3. Ciclo para generar los gráficos por cada ventana temporal
años_inicio = [2021, 2022, 2023, 2024, 2025]
orden_arima = (1, 1, 0)

print("Iniciando la generación de gráficos de líneas personalizados...")

for anio in años_inicio:
    print(f"Procesando visualizaciones para ventana: {anio} - 2026...")
    
    # Filtrar ventana temporal
    df_ventana = df_reducido[df_reducido['año'] >= anio].reset_index(drop=True)
    
    X_filtrada = df_ventana[columnas_exogenas]
    y = df_ventana['casos_dengue']
    fechas = df_ventana['fecha']
    
    # Reconstruir la estrategia Rolling Forecast para colectar las predicciones
    tamanio_entrenamiento_inicial = int(len(df_ventana) * 0.80)
    indices_test = range(tamanio_entrenamiento_inicial, len(df_ventana))
    
    predicciones_test = []
    mae_entrenamiento_lista = []
    
    # Simulación Rolling Forecast
    for i in indices_test:
        X_train_fold = X_filtrada.iloc[:i]
        y_train_fold = y.iloc[:i]
        X_test_fold = X_filtrada.iloc[i:i+1]
        
        modelo = ARIMA(endog=y_train_fold, exog=X_train_fold, order=orden_arima)
        modelo_ajustado = modelo.fit(method_kwargs={'maxiter': 150})
        
        # Guardar MAE entrenamiento local
        pred_train_fold = modelo_ajustado.fittedvalues
        mae_train_paso = mean_absolute_error(y_train_fold[1:], pred_train_fold[1:])
        mae_entrenamiento_lista.append(mae_train_paso)
        
        # Predecir paso adelante
        pred_test_paso = modelo_ajustado.forecast(steps=1, exog=X_test_fold)
        predicciones_test.append(pred_test_paso.values[0])
    
    # Calcular Métricas Finales de esta ventana para las leyendas
    mae_train_final = np.mean(mae_entrenamiento_lista)
    y_test_real = y.iloc[tamanio_entrenamiento_inicial:].values
    predicciones_test = np.array(predicciones_test)
    mae_test_final = mean_absolute_error(y_test_real, predicciones_test)
    
    # Obtener vectores correspondientes para graficar entrenamiento completo de la ventana
    fechas_train = fechas.iloc[:tamanio_entrenamiento_inicial]
    y_train_real = y.iloc[:tamanio_entrenamiento_inicial]
    fechas_test = fechas.iloc[tamanio_entrenamiento_inicial:]
    
    # Modelo final entrenado con el bloque completo de entrenamiento para la línea ajustada inicial
    modelo_grafico = ARIMA(endog=y_train_real, exog=X_filtrada.iloc[:tamanio_entrenamiento_inicial], order=orden_arima).fit(method_kwargs={'maxiter': 150})
    y_train_pred = modelo_grafico.fittedvalues

    # =============================================================================
    # DISEÑO DE LA GRÁFICA DE LÍNEAS (Subplots: Entrenamiento vs Testeo)
    # =============================================================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))
    fig.suptitle(f"Análisis de Líneas - Ventana Temporal ({anio} a 2026)", fontsize=14, fontweight='bold', y=0.98)
    
    # --- PANEL 1: CONJUNTO DE ENTRENAMIENTO ---
    ax1.plot(fechas_train, y_train_real, color='#1f77b4', label='Casos Reales Observados', alpha=0.75, linewidth=1.8)
    ax1.plot(fechas_train, y_train_pred, color='#ff7f0e', linestyle='--', 
             label=f'Ajuste ARIMAX (MAE: {mae_train_final:.2f})', alpha=0.85, linewidth=1.5)
    
    ax1.set_title("Conjunto de Entrenamiento (Histórico)", fontsize=11, fontweight='semibold')
    ax1.set_xlabel("Fecha", fontsize=10)
    ax1.set_ylabel("Casos de Dengue", fontsize=10)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none', shadow=True)
    ax1.tick_params(axis='x', rotation=35)
    
    # --- PANEL 2: CONJUNTO DE TESTEO (ROLLING FORECAST) ---
    ax2.plot(fechas_test, y_test_real, color='#2ca02c', marker='o', markersize=4, label='Casos Reales Observados', alpha=0.8)
    ax2.plot(fechas_test, predicciones_test, color='#d62728', marker='x', markersize=4, linestyle='-.',
             label=f'Predicción Dinámica (MAE: {mae_test_final:.2f})', alpha=0.9)
    
    ax2.set_title("Conjunto de Testeo (Validación Móvil)", fontsize=11, fontweight='semibold')
    ax2.set_xlabel("Fecha", fontsize=10)
    ax2.set_ylabel("Casos de Dengue", fontsize=10)
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='none', shadow=True)
    ax2.tick_params(axis='x', rotation=35)
    
    # Optimizar espaciado y guardar
    plt.tight_layout()
    nombre_archivo = f"lineas_desempeno_ventana_{anio}_2026.png"
    ruta_guardado = os.path.join(directorio_resultados, nombre_archivo)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"[OK] Gráfico guardado: {ruta_guardado}")

print("\n¡Visualizaciones completadas con éxito! Todos los archivos PNG se encuentran en la carpeta de resultados.")

Iniciando la generación de gráficos de líneas personalizados...
Procesando visualizaciones para ventana: 2021 - 2026...


findfont: Failed to find font weight semibold, now using 700.


[OK] Gráfico guardado: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\lineas_desempeno_ventana_2021_2026.png
Procesando visualizaciones para ventana: 2022 - 2026...
[OK] Gráfico guardado: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\lineas_desempeno_ventana_2022_2026.png
Procesando visualizaciones para ventana: 2023 - 2026...
[OK] Gráfico guardado: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\lineas_desempeno_ventana_2023_2026.png
Procesando visualizaciones para ventana: 2024 - 2026...
[OK] Gráfico guardado: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\lineas_desempeno_ventana_2024_2026.png
Procesando visualizaciones para ventana: 2025 - 2026...
[OK] Gráfico guardado: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\lineas_desempeno_ventana_2025_2026.png

¡Visualizaciones completadas con éxito! Todos los archivos PNG se encuentran en la carpeta de resultados.
